# Ball Tracking E2E — YOLO11


In [1]:
%pip install -q ultralytics roboflow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import csv
import os
from pathlib import Path

import torch
import yaml
from roboflow import Roboflow
from ultralytics import YOLO

PROJECT_ROOT = Path(r"D:\TRAINING\ball-tracker-cv").resolve()
assert PROJECT_ROOT.exists(), f"Root project tidak ditemukan: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
ENV_PATH = PROJECT_ROOT / ".env.example"
WORKSPACE = "xaviertheofiluss-workspace"
PROJECT_QUERY = "football player ball"
DATASET_VERSION = 1
MODEL_NAME = "yolo11n.pt"
EPOCHS = 50
IMAGE_SIZE = 640
DEVICE = "cpu"
BATCH_SIZE = 4

assert ENV_PATH.exists(), f"File tidak ditemukan: {ENV_PATH}"
env_lines = ENV_PATH.read_text(encoding="utf-8").splitlines()
API_KEY = next((
    line.split("=", 1)[1].strip().strip("\"'")
    for line in env_lines
    if line.strip().startswith("ROBOFLOW_API_KEY=")
), "")
assert API_KEY, "ROBOFLOW_API_KEY tidak ditemukan di .env.example"

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Model:", MODEL_NAME, "| Image size:", IMAGE_SIZE)

Project root: D:\TRAINING\ball-tracker-cv
Device: cpu
Model: yolo11n.pt | Image size: 640


In [2]:
rf = Roboflow(api_key=API_KEY)
workspace = rf.workspace(WORKSPACE)
available_projects = [project_id.rsplit("/", 1)[-1] for project_id in workspace.projects()]
query_slug = PROJECT_QUERY.strip().lower().replace(" ", "-")
PROJECT_ID = next((project_id for project_id in available_projects if project_id.lower() == query_slug), None)
if PROJECT_ID is None:
    PROJECT_ID = next((project_id for project_id in available_projects if project_id.lower().startswith(query_slug)), None)
assert PROJECT_ID is not None, (
    f"Project '{PROJECT_QUERY}' tidak ditemukan. Project tersedia: {available_projects}"
)
print("Roboflow project ID:", PROJECT_ID)

project = workspace.project(PROJECT_ID)
dataset = project.version(DATASET_VERSION).download(
    "yolov11",
    location=str(PROJECT_ROOT / "dataset"),
)

DATASET_DIR = Path(dataset.location).resolve()
DATA_YAML = DATASET_DIR / "data.yaml"
assert DATA_YAML.exists(), f"data.yaml tidak ditemukan: {DATA_YAML}"

data_config = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
downloaded_version = data_config.get("roboflow", {}).get("version")
assert downloaded_version == DATASET_VERSION, (
    f"Dataset salah: meminta v{DATASET_VERSION}, tetapi mendapat v{downloaded_version}"
)
readme_path = DATASET_DIR / "README.roboflow.txt"
readme_text = readme_path.read_text(encoding="utf-8", errors="ignore").lower() if readme_path.exists() else ""
for preprocessing_warning in ("grayscale", "stretch", "adaptive equalization"):
    if preprocessing_warning in readme_text:
        print(f"WARNING: dataset memakai {preprocessing_warning}; akurasi live dapat berkurang.")
raw_names = data_config.get("names", [])
CLASS_NAMES = list(raw_names.values()) if isinstance(raw_names, dict) else list(raw_names)
normalized_names = {name.strip().lower() for name in CLASS_NAMES}
assert normalized_names == {"ball", "player"}, (
    f"Class harus ball dan player, tetapi ditemukan: {CLASS_NAMES}"
)

print("Dataset:", DATASET_DIR)
print("Classes:", CLASS_NAMES)
for split in ("train", "valid", "test"):
    count = len(list((DATASET_DIR / split / "images").glob("*")))
    print(f"{split}: {count} images")

unique_train_images = {}
for image_path in sorted((DATASET_DIR / "train" / "images").glob("*")):
    source_frame = image_path.name.split(".rf.", 1)[0]
    unique_train_images.setdefault(source_frame, image_path.resolve())

CPU_TRAIN_LIST = DATASET_DIR / "train_cpu.txt"
CPU_TRAIN_LIST.write_text(
    "\n".join(str(path) for path in unique_train_images.values()),
    encoding="utf-8",
)
cpu_data_config = dict(data_config)
cpu_data_config["path"] = str(DATASET_DIR)
cpu_data_config["train"] = str(CPU_TRAIN_LIST)
cpu_data_config["val"] = str((DATASET_DIR / "valid" / "images").resolve())
cpu_data_config["test"] = str((DATASET_DIR / "test" / "images").resolve())
CPU_DATA_YAML = DATASET_DIR / "data_cpu.yaml"
CPU_DATA_YAML.write_text(yaml.safe_dump(cpu_data_config, sort_keys=False), encoding="utf-8")
DATA_YAML = CPU_DATA_YAML
estimated_batches = (len(unique_train_images) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"CPU training subset: {len(unique_train_images)} unique frames, ~{estimated_batches} batches/epoch")

loading Roboflow workspace...
Roboflow project ID: football-player-ball-ptjqv
loading Roboflow project...



Extracting Dataset Version Zip to D:\TRAINING\ball-tracker-cv\dataset in yolov11:: 100%|██████████| 4929/4929 [00:08<00:00, 556.77it/s]


Dataset: D:\TRAINING\ball-tracker-cv\dataset
Classes: ['ball', 'player']
train: 2004 images
valid: 229 images
test: 229 images
CPU training subset: 595 unique frames, ~149 batches/epoch


## Training

In [3]:
RUNS_ROOT = (PROJECT_ROOT / "runs").resolve()
model = YOLO(MODEL_NAME)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=0 if os.name == "nt" else 8,
    project=str(RUNS_ROOT),
    name="ball_detector_cpu_fast",
    exist_ok=True,
    optimizer="auto",
    patience=10,
    cos_lr=True,
    warmup_epochs=3,
    seed=42,
    deterministic=False,
    freeze=10,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
    plots=True,
)

BEST_WEIGHTS = Path(model.trainer.best).resolve()
assert BEST_WEIGHTS.exists(), f"best.pt tidak ditemukan: {BEST_WEIGHTS}"
print("Best model:", BEST_WEIGHTS)

New https://pypi.org/project/ultralytics/8.4.118 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.117  Python-3.12.13 torch-2.12.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\TRAINING\ball-tracker-cv\dataset\data_cpu.yaml, degrees=0.0, deterministic=False, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.0, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max

In [4]:
best_model = YOLO(str(BEST_WEIGHTS))
evaluation_split = "test" if data_config.get("test") else "val"
metrics = best_model.val(
    data=str(DATA_YAML),
    split=evaluation_split,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    workers=0 if os.name == "nt" else 8,
    project=str(RUNS_ROOT),
    name="evaluation_cpu_fast",
)

print("Evaluation split:", evaluation_split)
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP75:    {metrics.box.map75:.4f}")

Ultralytics 8.4.117  Python-3.12.13 torch-2.12.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 72.814.3 MB/s, size: 65.5 KB)
val: Scanning D:\TRAINING\ball-tracker-cv\dataset\test\labels... 229 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 229/229 1.3Kit/s 0.2s<0.3s
val: New cache created: D:\TRAINING\ball-tracker-cv\dataset\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.4s/it 36.7s2.3ss
                   all        229       2741      0.744      0.553       0.59      0.335
                  ball        192        192      0.598      0.286      0.328      0.121
                player        229       2549       0.89      0.819      0.852      0.548
Speed: 1.4ms preprocess, 137.7ms inference, 0.0ms loss, 5.3ms postprocess per image
Results saved to D:\TRAINING\b

## Tracking video


In [5]:
import csv
from pathlib import Path
from IPython.display import Video, display

# GANTI PATH INI SAJA
VIDEO_PATH = Path(
    r"D:\TRAINING\ball-tracker-cv\test2.mp4"
).resolve()

assert VIDEO_PATH.exists(), f"Video tidak ditemukan: {VIDEO_PATH}"

model_names = best_model.names
if isinstance(model_names, list):
    model_names = dict(enumerate(model_names))

BALL_CLASS_ID = next(
    class_id
    for class_id, name in model_names.items()
    if name.strip().lower() == "ball"
)

OUTPUT_ROOT = (
    Path(r"D:\TRAINING\ball-tracker-cv") / "outputs"
).resolve()

RUN_NAME = f"{VIDEO_PATH.stem}_tracking"

trajectory_rows = []
last_result = None
detected_frames = 0

results = best_model.track(
    source=str(VIDEO_PATH),
    conf=0.15,
    iou=0.50,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    tracker="bytetrack.yaml",
    persist=True,
    stream=True,
    save=True,
    project=str(OUTPUT_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    verbose=False,
)

for frame_number, result in enumerate(results):
    last_result = result
    ball_detections = []

    if result.boxes is not None:
        for box in result.boxes:
            if int(box.cls[0]) == BALL_CLASS_ID:
                x, y, _, _ = box.xywh[0].cpu().tolist()
                confidence = float(box.conf[0])

                ball_detections.append(
                    (confidence, x, y)
                )

    if ball_detections:
        confidence, x, y = max(ball_detections)

        trajectory_rows.append(
            (
                frame_number,
                f"{x:.2f}",
                f"{y:.2f}",
                f"{confidence:.4f}",
                "detected",
            )
        )

        detected_frames += 1

    else:
        trajectory_rows.append(
            (
                frame_number,
                "",
                "",
                "0.0000",
                "missing",
            )
        )

assert last_result is not None, "Video tidak menghasilkan frame"

SAVE_DIR = Path(last_result.save_dir).resolve()
CSV_PATH = SAVE_DIR / "trajectory.csv"

with CSV_PATH.open(
    "w",
    newline="",
    encoding="utf-8",
) as file:
    writer = csv.writer(file)

    writer.writerow(
        (
            "frame_number",
            "x_position",
            "y_position",
            "confidence",
            "status",
        )
    )

    writer.writerows(trajectory_rows)

video_files = [
    path
    for path in SAVE_DIR.iterdir()
    if path.suffix.lower() in {
        ".mp4",
        ".avi",
        ".mov",
    }
]

assert video_files, (
    f"Output video tidak ditemukan di {SAVE_DIR}"
)

OUTPUT_VIDEO = max(
    video_files,
    key=lambda path: path.stat().st_mtime,
)

coverage = (
    detected_frames / len(trajectory_rows)
    if trajectory_rows
    else 0
)

print("Input video:   ", VIDEO_PATH)
print("Output video:  ", OUTPUT_VIDEO)
print("Trajectory CSV:", CSV_PATH)
print(
    f"Ball coverage:  "
    f"{detected_frames}/{len(trajectory_rows)} "
    f"({coverage:.1%})"
)

display(
    Video(
        str(OUTPUT_VIDEO),
        embed=False,
    )
)

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: d:\Users\bsi80274\.conda\envs\ai-training
Resolved 2 packages in 604ms
 Downloaded lap
Prepared 1 package in 586ms
Installed 1 package in 18ms
 + lap==0.5.13

requirements: AutoUpdate success  1.9s
WARNING requirements: Restart runtime or rerun command for updates to take effect

Results saved to D:\TRAINING\ball-tracker-cv\outputs\test2_tracking
Input video:    D:\TRAINING\ball-tracker-cv\test2.mp4
Output video:   D:\TRAINING\ball-tracker-cv\outputs\test2_tracking\test2.avi
Trajectory CSV: D:\TRAINING\ball-tracker-cv\outputs\test2_tracking\trajectory.csv
Ball coverage:  486/1500 (32.4%)
